In [56]:
import tensorflow as tf
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

In [57]:
df = pd.read_csv('creditcard.csv')
print(df.shape)
print(df.columns)

(284807, 31)
Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')


Scaling the Data SET

In [58]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1,1))
df['Time'] = scaler.fit_transform(df['Time'].values.reshape(-1,1))

In [59]:
X = df.drop(columns=['Class'])
print(X.shape)
Y = df.Class

(284807, 30)


In [60]:
X_train , X_ , Y_train , Y_ = train_test_split(X,Y,test_size=0.4 , random_state=42)
X_cv , X_test , Y_cv , Y_test = train_test_split(X_,Y_,test_size=0.5 , random_state = 42)

In [61]:
model = Sequential([
    Dense(256 , activation = 'relu') ,
    Dense(128,activation='relu') ,
    Dense(64 , activation = 'relu') ,
    Dense(16 , activation = 'relu') ,
    Dense(1 , activation = 'sigmoid')
])

model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(0.005)
)

Using Class Weights

In [62]:
from sklearn.utils import class_weight

weights = class_weight.compute_class_weight(
    class_weight='balanced' ,
    classes = np.unique(Y_train) ,
    y = Y_train
)

class_weights = {0: weights[0] , 1:weights[1]}
print(f"Class weights : {class_weights}")

Class weights : {0: np.float64(0.5008822684558251), 1: np.float64(283.86046511627904)}


In [63]:
model.fit(X_train , Y_train , epochs=10 , batch_size=2048 , class_weight = class_weights)

Epoch 1/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - loss: 0.2977
Epoch 2/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.1537
Epoch 3/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.1441
Epoch 4/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - loss: 0.0919
Epoch 5/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.1027
Epoch 6/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0789
Epoch 7/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0595
Epoch 8/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0947
Epoch 9/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0934
Epoch 10/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.1005


In [64]:
def err_cat(y , yhat):
  incorrect = 0
  m = y.shape[0]

  for i in range(m):
    if (yhat[i] != y[i]):
      incorrect += 1
  err = incorrect / m
  return err

Finding Best Threshold value

In [81]:
from sklearn.metrics import f1_score

y_prob = model.predict(X_test)
best_f1 = 0
best_threshold = 0
threshold = np.arange(0.1 , 1.0 , 0.001)

for thresh in threshold:
  y_pred = (y_prob >= thresh).astype(int)
  f1 = f1_score(Y_test , y_pred)
  if f1 > best_f1 :
    best_f1 = f1
    best_threshold = thresh


print(f'Best Threshold = {best_threshold}')

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Best Threshold = 0.9990000000000008


In [82]:
y_train_pred_probs = model.predict(X_train)
y_train_pred_labels = (y_train_pred_probs >= best_threshold).astype(int)
J_train = err_cat(Y_train.values , y_train_pred_labels)

y_cv_pred_probs = model.predict(X_cv)
y_cv_pred_labels = (y_cv_pred_probs >= best_threshold).astype(int)
J_cv = err_cat(Y_cv.values , y_cv_pred_labels)

print(f"J_train = {J_train} \n")
print(f"J_cv = {J_cv}")

5341/5341 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step
1781/1781 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
J_train = 0.0006612672924322932 

J_cv = 0.0006495672477660154


In [79]:
yhat_prob = model.predict(X_test)
yhat_labels = (yhat_prob >= best_threshold).astype(int)
J_test = err_cat(Y_test.values , yhat_labels)

print(f"J_test %  = {(J_test * 100)} % ")

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
J_test %  = 0.07548892243952109 % 


In [80]:
from sklearn.metrics import classification_report

print("--------Classification report ------------")
print(classification_report(Y_test , yhat_labels , target_names=['Normal' , 'Fraud']))

--------Classification report ------------
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56869
       Fraud       0.74      0.84      0.78        93

    accuracy                           1.00     56962
   macro avg       0.87      0.92      0.89     56962
weighted avg       1.00      1.00      1.00     56962

